<a target="_blank" href="https://colab.research.google.com/github/takurot/crypto-rs-backtester/blob/main/example/colab_backtester_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# crypto-rs-backtester: Google Colab Demo

This notebook demonstrates building and using the Rust-PyO3 backtester from Colab, showcasing features and basic performance characteristics.

What you'll see:
- Build the PyO3 extension via maturin
- Minimal end-to-end smoke check
- Tick vs Batch processing equivalence
- Strategy callback with feed latency
- Micro-benchmark of batched callbacks
- (Optional) end-to-end throughput on synthetic data


In [ ]:
# Install Rust toolchain (idempotent)
import os, shutil, subprocess, sys
if not shutil.which("rustc"):
    print("Installing Rust toolchain via rustup...")
    subprocess.run("curl https://sh.rustup.rs -sSf | sh -s -- -y", shell=True, check=True)
    os.environ["PATH"] += f":{os.path.expanduser('~')}/.cargo/bin"
print(subprocess.check_output("rustc --version", shell=True, text=True))


In [ ]:
%pip -q install -U maturin polars pytest


In [ ]:
# Clone the repository and build/install the Python package.
import os, subprocess, shutil
REPO_URL = os.environ.get("BACKTESTER_REPO", "https://github.com/takurot/crypto-rs-backtester.git")
BRANCH = os.environ.get("BACKTESTER_BRANCH", "main")
REPO_DIR = '/content/crypto-rs-backtester'

if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_URL} ...')
    subprocess.run(f'git clone {REPO_URL} {REPO_DIR}', shell=True, check=True)
    # Try to checkout the requested branch; fall back silently.
    r = subprocess.run(f'cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH}', shell=True)
    if r.returncode != 0:
        print(f'Branch {BRANCH} not found on remote; using default branch.')
else:
    # Pull updates if repo already exists.
    subprocess.run(f'cd {REPO_DIR} && git fetch && git pull --ff-only', shell=True, check=False)

# Ensure cargo is on PATH for maturin
os.environ["PATH"] += f":{os.path.expanduser('~')}/.cargo/bin"

print('Building and installing rust_backtester...')
# Note: We use standard install (not -e) to ensure availability in Colab without restart.
r = subprocess.run('python -m pip -q install .[dev]', cwd=REPO_DIR, shell=True)
if r.returncode != 0:
    print('pip install failed, trying maturin develop --release (fallback)...')
    subprocess.run('maturin develop --release -m pyproject.toml', cwd=REPO_DIR, shell=True, check=True)

# Verify import in current process to catch path issues early
try:
    import rust_backtester as rb
    print(f'Import successful: {rb.__version__}')
except ImportError:
    print('Module not found immediately. Triggering site reload...')
    import site
    import importlib
    importlib.reload(site)

print('Installed version (subprocess check):')
print(subprocess.check_output("python -c 'import rust_backtester as rb; print(rb.__version__)'", shell=True, text=True))


## 1) Minimal smoke (FFI + determinism plumbing)


In [ ]:
import polars as pl
import rust_backtester as rb

lf = pl.DataFrame({
    'ts_exchange': [1000, 2000, 3000, 4000],
    'price': [100_00000000, 101_00000000, 99_00000000, 100_00000000],
    'qty': [1_00000000, 1_00000000, 1_00000000, 1_00000000],
    'side': [1, -1, 1, -1],
    'seq': [0, 1, 2, 3],
}).lazy()

bt = rb.Backtester(data={'binance:BTC/USDT': lf}, seed=42)
checksum = bt.run_smoke()
checksum


## 2) Tick-mode strategy with feed latency
Demonstrates per-tick callback and latency-applied delivery timestamps.


In [ ]:
class RecordingStrategy:
    def __init__(self):
        self.ticks = []
    def on_tick(self, tick: dict, ctx):  # noqa: ANN001
        self.ticks.append(tick)

feed_latency_ns = 1_000
bt = rb.Backtester(
    data={'binance:BTC/USDT': lf},
    seed=42,
    python_mode='tick',
    batch_ms=100,
    feed_latency_ns=feed_latency_ns,
)
strat = RecordingStrategy()
_ = bt.run(strat)
[ (t['ts_exchange'], t['ts_local']) for t in strat.ticks ]


## 3) Tick vs Batch equivalence (determinism)
Runs the same dataset through both modes and compares core stats.


In [ ]:
bt_tick = rb.Backtester(data={'binance:BTC/USDT': lf}, seed=42, python_mode='tick')
bt_batch = rb.Backtester(data={'binance:BTC/USDT': lf}, seed=42, python_mode='batch')
res_tick = bt_tick.run(RecordingStrategy())
res_batch = bt_batch.run(RecordingStrategy())
stats_tick = res_tick.stats()
stats_batch = res_batch.stats()

{ 'tick': dict(stats_tick), 'batch': dict(stats_batch) }


## 4) Order submission and updates (preview)
Submit a simple limit order from Python and observe order updates (if any).
Note: matching depends on the queue model and tick data; this is a minimal plumbing demo.


In [ ]:
class SubmittingStrategy:
    def __init__(self):
        self.updates = []
        self._submitted = False
    def on_tick(self, tick: dict, ctx):  # noqa: ANN001
        if not self._submitted:
            # Submit a limit order at current price.
            ctx.submit_order(symbol_id=tick['symbol_id'], side=1, price=tick['price'], qty=1_00000000)
            self._submitted = True
    def on_order_update(self, report: dict, ctx):  # noqa: ANN001
        self.updates.append(report)

bt = rb.Backtester(data={'binance:BTC/USDT': lf}, seed=42, python_mode='tick', feed_latency_ns=1_000)
s = SubmittingStrategy()
_ = bt.run(s)
len(s.updates), (s.updates[0] if s.updates else None)


## 5) Batched Python callback micro-benchmark
Measures overhead of Python<->Rust boundary when batching ticks.


In [ ]:
import time

class Noop:
    def on_ticks(self, ticks):
        return None

noop = Noop()
batch_size = 1024
iterations = 2000

t0 = time.perf_counter()
rb.call_strategy_on_ticks(noop, batch_size, iterations)
t1 = time.perf_counter()
rust_loop_sec = t1 - t0

ticks = list(range(batch_size))
t0 = time.perf_counter()
for _ in range(iterations):
    noop.on_ticks(ticks)
t1 = time.perf_counter()
python_loop_sec = t1 - t0

total = batch_size * iterations
{
 'total_calls': total,
 'rust_wrapped_sec': round(rust_loop_sec, 4),
 'python_loop_sec': round(python_loop_sec, 4),
 'rust_wrapped_Mticks_per_s': round(total / 1e6 / rust_loop_sec, 2),
 'python_loop_Mticks_per_s': round(total / 1e6 / python_loop_sec, 2),
}


## 6) End-to-end throughput on synthetic data (optional)
Generates 200k ticks and times a backtest run (includes Python<->Rust data marshaling).


In [ ]:
import time
N = 200_000
df = pl.DataFrame({
    'ts_exchange': list(range(1, N+1)),
    'price': [100_00000000] * N,
    'qty': [1_00000000] * N,
    'side': [1 if (i % 2 == 0) else -1 for i in range(N)],
    'seq': list(range(N)),
})
lf_synth = df.lazy()
bt = rb.Backtester(data={'binance:BTC/USDT': lf_synth}, seed=1, python_mode='batch')
t0 = time.perf_counter()
res = bt.run(RecordingStrategy())
t1 = time.perf_counter()
elapsed = t1 - t0
{ 'rows': N, 'sec': round(elapsed, 3), 'Krows/s': round(N/elapsed/1e3, 1) }
